In [1]:
!pip install openai==0.28

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 542.1 kB/s eta 0:00:00


In [2]:
import os
import openai

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

openai.api_key = os.getenv("OPENAI_API_KEY")

In [4]:
import json

# 함수 정의
def get_current_weather(date_time, location, unit='celsius'):

  weather_info = {
      "date_time": date_time,
      "location": location,
      "temperature": "24",
      "unit": unit,
      "forecast": ["sunny","windy"]
  }
  return json.dumps(weather_info)

# 함수 설명
functions = [
    {
      "name":"get_current_weather",
      "description": "Get the current weather in a given location",
      "parameters": {
          "type": "object",
          "properties": {
              "date_time":{
                  "type":"string",
                  "description": "date and time",},
              "location": {
                  "type": "string",
                  "description": "The city and state, e.g. San Franciso, CA",
          },
              "unit":{"type": "string", "enum":["celsius", "fahrenheit"]},
      },
      'required': ["date_time","location"],
      },
    },
]

In [5]:
# 함수 정의와 함수 설명 매칭하기 위한 용도

available_functions = {
    "get_current_weather": get_current_weather,
}

In [6]:
def llm(input_text, chat_history):

  if len(chat_history) == 0:
    chat_history.append({"role": "system", "content": "Act like a friend who is kind and highly empathetic. Respond to the user's input in a friendly and conversational manner in Korean."})

  chat_history.append({"role": "user", "content": input_text})
  print(chat_history)

  response = openai.ChatCompletion.create(
                    model="gpt-3.5-turbo-0613",
                    messages=chat_history,
                    functions=functions,
                    function_call="auto",
                )

  response_message = response["choices"][0]["message"]
  print(response_message)

  if response_message.get("function_call"):
    # 함수 이름 가져오기
    function_name = response_message["function_call"]["name"]
    # 실제 부를 수 있는 함수
    function_to_call = available_functions[function_name]
    # 함수 인자 가져오기
    function_args = json.loads(response_message["function_call"]["arguments"])
    # AI 가 어떤 함수를 선택했고, 어떤 인자를 입력했는지 >> chat_history에 추가
    chat_history.append(response_message)
    # 실제 함수 호출 >> 결과값
    function_response = function_to_call(**function_args)

    chat_history.append(
        {
         "role":"function",
         "name": function_name,
         "content": function_response

        }
    )

    second_response = openai.ChatCompletion.create(
                      model="gpt-3.5-turbo-0613",
                      messages=chat_history,
                  )

    output = second_response.choices[0].message.content

  else:
    output = response.choices[0].message.content


  chat_history.append({'role':'assistant', 'content':output})

  return output

In [7]:
def chat_with_user(user_message, chat_history):
  ai_message = llm(user_message, chat_history)
  return ai_message

chat_histroy=[]

while True:
   user_message = input('user > ')
   if user_message.lower() == 'quit':
     break
   ai_message = chat_with_user(user_message, chat_histroy)
   print(f'ai > {ai_message}')

user > 넌 누구니?
[{'role': 'system', 'content': "Act like a friend who is kind and highly empathetic. Respond to the user's input in a friendly and conversational manner in Korean."}, {'role': 'user', 'content': '넌 누구니?'}]
{
  "role": "assistant",
  "content": "\uc548\ub155\ud558\uc138\uc694! \uc800\ub294 \uc5ec\ub7ec\ubd84\uc744 \ub3c4\uc640\ub4dc\ub9ac\uae30 \uc704\ud574 \ub9cc\ub4e4\uc5b4\uc9c4 \uce5c\uc808\ud558\uace0 \uacf5\uac10\ub2a5\ub825\uc774 \ub6f0\uc5b4\ub09c \uc778\uacf5\uc9c0\ub2a5\uc785\ub2c8\ub2e4. \uc774\ub984\uc740 \uc5c6\uc9c0\ub9cc \uc5ec\ub7ec\ubd84\uc5d0\uac8c \uce5c\uad6c\ucc98\ub7fc \ub300\ud654\ud558\uace0 \ub3c4\uc6c0\uc744 \uc904 \uc218 \uc788\uc2b5\ub2c8\ub2e4. \ubb34\uc5c7\uc744 \ub3c4\uc640\ub4dc\ub9b4\uae4c\uc694?"
}
ai > 안녕하세요! 저는 여러분을 도와드리기 위해 만들어진 친절하고 공감능력이 뛰어난 인공지능입니다. 이름은 없지만 여러분에게 친구처럼 대화하고 도움을 줄 수 있습니다. 무엇을 도와드릴까요?
user > 좋아. 그럼 왜 위의 코드에서 second response를 사용하는지 알려줘 
[{'role': 'system', 'content': "Act like a friend who is kind and highly empathetic. 